# V8: feature weight theo QP

Bật GPU + Internet. Add Input: Output V7 đầy đủ với `v7_rate_recovery_20260907_163149_952110/recovery_split.json`, `selection.json`, `proxy_calibrated/best.pt`; Output V6 có `v6_calibration_20260905_161558_046557/swin_ratio_090/best_task_bd_rate.pt` thực sự −6.687%; checking có `v5_fixed_split/split_manifest.json`; kineticscleaned.

Đường dẫn video: `/kaggle/input/datasets/qktttttttttt/kineticscleaned/cleaned_final/kinetics400_5per/kinetics400_5per/train`. File còn trong Working cũng dùng được. Giữ cấu trúc thư mục Output gốc khi thêm Input.

Run All: khôi phục train/controller V7, đo baseline, fine-tune 2 epoch rồi đánh giá full validation 7 QP bằng H.264 thật. Dùng lại proxy V7, khởi tạo Swin V6 trong run mới. Relative MSE layer3/layer4 với tỷ lệ 0.3/0.7, feature weights QP30/35/40/45 = 0.03/0.04/0.06/0.07, target BPP ratio 0.95, lr 1e-5. Hệ số QP là cấu hình thử nghiệm, chưa đảm bảo đạt BD-rate dưới −10%. Nếu chưa feasible, Cell 5 vẫn đánh giá và giữ nhãn chẩn đoán. Lỗi train sớm hoặc proxy guard sẽ dừng pipeline.


In [ ]:
# 1. Update code and import helpers
from pathlib import Path
import subprocess, sys, importlib

PROJECT = Path("/kaggle/working/proxy_v3")
if PROJECT.exists():
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/munnn01/proxy_v3.git", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")], check=True)
sys.path.insert(0, str(PROJECT))
importlib.invalidate_caches()
from preprocessing import feature_distillation
from kaggle_cells import v7_rate_recovery, v8_feature_ablation
importlib.reload(feature_distillation)
importlib.reload(v7_rate_recovery)
v8 = importlib.reload(v8_feature_ablation)



In [ ]:
# 2. Select inputs and per-QP feature coefficients
# Auto-find the original V6 checkpoint, V7 proxy, and exact V7 controller split.
RUN = v8.prepare()
FEATURE_BY_QP = {30: 0.03, 35: 0.04, 40: 0.06, 45: 0.07}
FEATURE_WEIGHTS = [FEATURE_BY_QP[q] for q in [30, 35, 40, 45]]
print("Feature weights:", FEATURE_BY_QP)
print("Swin input:", RUN.checkpoint)
print("Calibrated proxy input:", RUN.starting_proxy)
print("Output:", RUN.root)



In [ ]:
# 3. Measure the starting V6 checkpoint on the same 800-video controller
BASELINE_JSON = v8.baseline(RUN, feature_weights_by_qp=FEATURE_WEIGHTS)



In [ ]:
# 4. Fine-tune two epochs with a fresh optimizer and rate-dual controller
CANDIDATE_PT, IS_FEASIBLE = v8.train(RUN, feature_weights_by_qp=FEATURE_WEIGHTS)
print("Candidate:", CANDIDATE_PT, "Feasible:", IS_FEASIBLE)



In [ ]:
# 5. Compare on the controller, then evaluate real H.264 at all seven QPs
import json, torch
import pandas as pd
from IPython.display import display, FileLink

initial = json.loads(BASELINE_JSON.read_text())["val_metrics"]
payload = torch.load(CANDIDATE_PT, map_location="cpu", weights_only=False)
candidate = payload["val_metrics"]
print("Controller initial BD-rate (%):", initial["task_bd_rate_percent"])
print("Controller candidate BD-rate (%):", candidate["task_bd_rate_percent"])
print("Feasible:", IS_FEASIBLE, "Epoch:", payload["epoch"])
display(pd.DataFrame([
    {"QP": q, "feature_weight": candidate[f"qp{q}_feature_weight"],
     "initial_BPP_ratio": initial[f"qp{q}_bpp_ratio"],
     "candidate_BPP_ratio": candidate[f"qp{q}_bpp_ratio"],
     "initial_Top1_%": 100 * initial[f"qp{q}_top1"],
     "candidate_Top1_%": 100 * candidate[f"qp{q}_top1"]}
    for q in [30, 35, 40, 45]
]))
del payload
if not IS_FEASIBLE:
    print("Diagnostic checkpoint: BPP/Top-1 constraints were not met on the controller.")
EVAL_DIR = v7_rate_recovery.evaluate_candidate(RUN, CANDIDATE_PT)
for name in ("metrics.csv", "per_video_metrics.csv", "bd_rate.json"):
    display(FileLink(str(EVAL_DIR / name)))
